# Batch effect correction

In [62]:
import pandas as pd
from pygments.lexers.shell import BatchLexer

from src.filters import *
from src.utils import *
from src.column_spec import *

In [14]:
df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/dia-quant-output/abundance_multi-site_MS2quant_Norm.tsv", sep="\t", low_memory=False)
batch1_df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Queue/20260619_TIMSTOF_1_42386_batch1.csv", header=1)
batch2_df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Queue/20260625_TIMSTOF_1_42386_batch2.csv", header=1)
batch3_df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Queue/20260702_TIMSTOF_1_42386_batch3.csv", header=1)
df

,Index,Gene,ProteinID,Peptide,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260625_078_C42386_S1171756_132.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260702_076_C42386_S1171836_106.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260625_042_C42386_S1171708_114.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260702_025_C42386_S1171811_70.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260619_074_C42386_S1171348_184.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260625_035_C42386_S1171697_29.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260702_079_C42386_S1171837_134.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260619_016_C42386_S1171329_99.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260619_029_C42386_S1171291_136.d,E:\FGCZ\p31978\Workunit_348536_20260722_103347\resources\20260702_041_C42386_S1171787_61.d
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0000,P16333,...,1428.0177,NaN,NaN,NaN,NaN,3956.5876,NaN,3551.3633,NaN,NaN
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0000,Q9Y618,...,548.7228,NaN,NaN,1198.2482,NaN,1098.6317,NaN,NaN,545.8596,NaN
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0000,Q9H2G2,...,1085.7393,NaN,NaN,1590.3137,1489.6044,1688.5862,NaN,NaN,NaN,1451.7201
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0000,O15085,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1188.2247,NaN,427.1053
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0000,P08670,...,312.6730,NaN,NaN,NaN,536.3586,NaN,184.9900,289.6471,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71082,Q13112_507_538_1_1_S538,CHAF1B,Q13112,TDTPPSSVPTSVISTPSTEEIQSETPGDAQGSPPELKR;TDTPPSS...,RINLTPLK.TDTPPSSVPTSVISTPSTEEIQSETPGDAQGSPPELK...,507,543,37,1.0000,Q13112,...,73960.6172,107488.8203,NaN,97859.5000,98353.6563,116447.8906,95083.6484,133127.3750,137619.9375,103166.1953
71083,Q9H3Q1_9_14_1_1_S11,CDC42EP4,Q9H3Q1,QLVSSSVHSK,MPILK.QLVSSSVHSK.RRSRADLT,6,15,10,0.9997,Q9H3Q1,...,NaN,NaN,NaN,NaN,328.7932,NaN,NaN,NaN,290.2783,NaN
71084,Q13470_543_553_1_1_S543,TNK1,Q13470,AVPQGPPGLPPRPPLSSSSPQPSQPSR,PPEIRQAR.AVPQGPPGLPPRPPLSSSSPQPSQPSR.ERLPWPKR,528,554,27,0.9999,Q13470,...,6281.0952,3057.6907,NaN,5174.8745,4944.7759,3924.1655,7276.3989,3950.4485,11016.1572,5325.3438
71085,Q9UQ35_332_335_1_1_Y335,SRRM2,Q9UQ35,QPSSPYEDKDKDK;QPSSPYEDKDK,SSPETATK.QPSSPYEDKDKDK.KEKSATRP,330,342,13,1.0000,Q9UQ35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
CELL_LINES = ["WT", "EGFRT693A", "BRAFS151A1", "SOS1S1178A", "SHOC2T71A", "BRAFS151A2", "GAB1Y259A", "RPS6KA3S375A"]
TIME_POINTS = ["full", "starve", "2", "5", "10", "15", "20", "30", "90"]
REPLICATES = ["r1", "r2", "r3"]
CONDITION = "EGF" #"_EGF_"
DATA_TYPE = "raw:abs"

# Cell lines naming dictionary
labels_dic = {}
c = 1
for cell in CELL_LINES:
    for tp in TIME_POINTS:
        for rep in REPLICATES:
            labels_dic[str(c)] = cell + "_" + DATA_TYPE + "_" + CONDITION + "_" + tp + "_" + rep
            c += 1

# Control channels naming
MIX_LABELS = {"mix":  "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r1",
              "mixb": "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r2",
              "mixc": "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r3",}

In [56]:
def batch_loading_order(info_col:list,
                        batch_number:int,
                        labels_dic:dict,
                        mix_labels_dic:dict, ):
    """

    """
    batch_order = {}
    for element in info_col:
        key = int(str(batch_number) + str(element.split("_")[1]))
        sample =  str(element.split("_")[-1])
        if sample in labels_dic.keys():
            sample_name = labels_dic[sample]
            batch_order[key] = sample_name
        elif sample in mix_labels_dic.keys():
            sample_name = mix_labels_dic[sample]
            batch_order[key] = sample_name

    return batch_order


info_col1 = batch1_df["/Sample/@SampleID"].tolist()
info_col2 = batch2_df["/Sample/@SampleID"].tolist()
info_col3 = batch3_df["/Sample/@SampleID"].tolist()

batch_1 = batch_loading_order(info_col= info_col1, batch_number=1, labels_dic= labels_dic, mix_labels_dic= MIX_LABELS)
batch_2 = batch_loading_order(info_col= info_col2, batch_number=2, labels_dic= labels_dic, mix_labels_dic= MIX_LABELS)
batch_3 = batch_loading_order(info_col= info_col3, batch_number=3, labels_dic= labels_dic, mix_labels_dic= MIX_LABELS)


In [89]:
def fill_info_df(batch_dir:dir):

    batch = batch_dir
    info_df = pd.DataFrame(columns=["sample_Id", "cell_line", "condition", "timepoint", "batch", "is_reference", "injection_order"])

    info_df["sample_Id"] = batch.values()
    info_df["cell_line"] = pd.Series(list(batch.values())).str.split("_").str[0]
    info_df["condition"] = pd.Series(list(batch.values())).str.split("_").str[2]
    info_df["timepoint"] = pd.Series(list(batch.values())).str.split("_").str[3]
    info_df["batch"] = pd.Series(list(batch.keys())).astype(str).str[0]
    info_df["injection_order"] = pd.Series(list(batch.keys())).astype(str).str[1:].astype(int)

    for index, row in info_df.iterrows():
        if row["timepoint"] == "full":
            info_df.loc[index, "condition"] = "full"
            info_df.loc[index, "timepoint"] = 0
        if row["timepoint"] == "starve":
            info_df.loc[index, "condition"] = "starve"
            info_df.loc[index, "timepoint"] = 0
        if row["sample_Id"].startswith("MIX"):
            info_df.loc[index, "cell_line"] = "reference"
            info_df.loc[index, "condition"] = "reference"
            info_df.loc[index, "timepoint"] = 0
        if row["sample_Id"].startswith("MIX"):
            info_df.loc[index, "is_reference"] = "TRUE"
        else:
            info_df.loc[index, "is_reference"] = "FALSE"

    return info_df

df_batch1 = fill_info_df(batch_1)
df_batch2 = fill_info_df(batch_2)
df_batch3 = fill_info_df(batch_3)

df = pd.concat([df_batch1, df_batch2, df_batch3], ignore_index=True)
df

,sample_Id,cell_line,condition,timepoint,batch,is_reference,injection_order
0,RPS6KA3S375A_raw:abs_EGF_full_r3,RPS6KA3S375A,full,0,1,FALSE,3
1,EGFRT693A_raw:abs_EGF_2_r3,EGFRT693A,EGF,2,1,FALSE,4
2,SHOC2T71A_raw:abs_EGF_2_r1,SHOC2T71A,EGF,2,1,FALSE,5
3,SOS1S1178A_raw:abs_EGF_30_r3,SOS1S1178A,EGF,30,1,FALSE,6
4,EGFRT693A_raw:abs_EGF_starve_r3,EGFRT693A,starve,0,1,FALSE,7
...,...,...,...,...,...,...,...
214,SOS1S1178A_raw:abs_EGF_15_r1,SOS1S1178A,EGF,15,3,FALSE,75
215,SOS1S1178A_raw:abs_EGF_90_r1,SOS1S1178A,EGF,90,3,FALSE,76
216,WT_raw:abs_EGF_starve_r3,WT,starve,0,3,FALSE,77
217,BRAFS151A2_raw:abs_EGF_20_r3,BRAFS151A2,EGF,20,3,FALSE,78


In [90]:
df.to_csv("../../Experiment/hme1_diaPASEF/Data/batch_info.tsv", sep="\t", index=False)